In [1]:
# Utils
# =============================================================================
from utils.paths import (
    ORDERS_FILE, 
    ORDER_ITEMS_FILE, 
    PAYMENTS_FILE, 
    REVIEWS_FILE, 
    CUSTOMERS_FILE, 
    PRODUCTS_FILE, 
    SELLERS_FILE, 
    GEOLOCATION_FILE, 
    CATEGORY_TRANSLATION_FILE
)

# DATA
# =============================================================================
import pandas as pd


In [2]:
orders = pd.read_csv(ORDERS_FILE)
payments_file = pd.read_csv(PAYMENTS_FILE)

In [3]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

In [4]:
orders_payments = pd.merge(orders, payments_file, on='order_id', how='left')


In [5]:
orders['customer_id'].shape

(99441,)

In [6]:
function_dictionary = {
    "payment_value": ['sum'],
}
filter_feature = ['order_id']
new_val_col = [
    'order_id',
    'sum_payment'
]
payments_group = payments_file.groupby(
    filter_feature).aggregate(function_dictionary)
payments_group = payments_group.reset_index()
payments_group.columns = new_val_col
payments_group.head()

,order_id,sum_payment
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [7]:
orders_payments = pd.merge(
    orders,
    payments_group,
    on='order_id',
    how='inner'
)

In [9]:
function_dictionary = {
    "order_id": ['nunique'],
    "sum_payment": ['sum', 'mean', 'median'],
}

filter_feature = [
    'order_purchase_timestamp']

new_val_col = [
    'order_purchase_timestamp',
    'count_orders',
    'sum_payment',
    'mean_payment',
    'median_payment'
]

data_group = orders_payments.groupby(
    pd.Grouper(key='order_purchase_timestamp', freq='ME')).aggregate(function_dictionary)

data_group = data_group.reset_index()
data_group.columns = new_val_col
data_group

,order_purchase_timestamp,count_orders,sum_payment,mean_payment,median_payment
0,2016-09-30,3,252.24,84.080000,75.060
1,2016-10-31,324,59090.48,182.378025,112.650
2,2016-11-30,0,0.00,NaN,NaN
3,2016-12-31,1,19.62,19.620000,19.620
4,2017-01-31,800,138488.04,173.110050,106.130
5,2017-02-28,1780,291908.01,163.993264,105.155
6,2017-03-31,2682,449863.60,167.734377,102.640
7,2017-04-30,2404,417788.03,173.788698,104.350
8,2017-05-31,3700,592918.82,160.248330,104.450
9,2017-06-30,3245,511276.38,157.558206,100.820


In [ ]:
# VISUALIZATION
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid')


def plot_monthly_line(data, y_col, title, y_label, color):
    """Line chart of a monthly metric against order_purchase_timestamp."""
    fig, ax = plt.subplots(figsize=(12, 4.5))
    sns.lineplot(
        data=data,
        x='order_purchase_timestamp',
        y=y_col,
        marker='o',
        color=color,
        ax=ax
    )
    ax.set_title(title, fontsize=13, loc='left')
    ax.set_xlabel('Mes')
    ax.set_ylabel(y_label)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    fig.tight_layout()
    plt.show()

In [ ]:
plot_monthly_line(
    data_group,
    y_col='count_orders',
    title='Número de pedidos por mes',
    y_label='Pedidos (count_orders)',
    color='#1f77b4'
)

In [ ]:
plot_monthly_line(
    data_group,
    y_col='sum_payment',
    title='Monto total pagado por mes',
    y_label='Monto pagado (sum_payment)',
    color='#2ca02c'
)

In [ ]:
plot_monthly_line(
    data_group,
    y_col='mean_payment',
    title='Valor promedio por pedido (AOV) por mes',
    y_label='AOV (mean_payment)',
    color='#ff7f0e'
)


## Patrones temporales observados

Descripción de lo observado en los datos de `data_group`; no se interpretan causas.

**`count_orders` (pedidos por mes)**
- Tendencia: crece de 3 pedidos (2016-09) a 4.631 (2017-10) y 7.544 (2017-11, máximo de la serie), con bajas puntuales en 2017-04 y 2017-06.
- Cambios bruscos: 2017-11 sube frente a 2017-10 y en 2017-12 baja a 5.673; entre 2018-08 y 2018-09 la serie cae de 6.512 a 16 pedidos (4 en 2018-10).
- Estabilidad: entre 2018-01 y 2018-08 los valores se mantienen en el rango 6.167 – 7.269.
- Ciclos/estacionalidad: solo hay un pico en noviembre (2017-11); con menos de dos años completos no se puede confirmar un ciclo anual.
- Cambios relevantes: 2016-11 registra 0 pedidos y 2016-12 registra 1; 2016-09/10 y 2018-09/10 son meses con muy pocos registros en los datos.

**`sum_payment` (monto total pagado por mes)**
- Tendencia: crece de forma general hasta 2017-11 (≈1,19 M, máximo de la serie), con bajas puntuales en 2017-04 y 2017-06.
- Cambios bruscos: mismo patrón que `count_orders`: pico en 2017-11, baja en 2017-12 y caída final en 2018-09 y 2018-10.
- Estabilidad: entre 2018-01 y 2018-08 se mantiene entre ≈0,99 M y ≈1,16 M.
- Ciclos/estacionalidad: la forma de la serie es muy parecida a la de `count_orders`; aplica la misma limitación de un solo pico observado.
- Cambios relevantes: 2016-11 tiene monto 0 y los meses de los extremos de la serie tienen montos muy bajos.

**`mean_payment` (AOV mensual)**
- Tendencia: no muestra una tendencia clara; oscila aproximadamente entre 147 y 174 durante 2017-01 a 2018-08.
- Cambios bruscos: 2016-11 no tiene valor (sin pedidos, `NaN`, se ve como un corte en la línea); 2018-09 marca 277,47 con solo 16 pedidos, un valor fuera del rango del resto de la serie. 2016-10 (182,38) también queda por encima de ese rango.
- Estabilidad: es la métrica más estable de las tres en los meses con volumen alto.
- Ciclos/estacionalidad: hay pequeñas subidas y bajadas mes a mes, sin un patrón repetido evidente.
- Cambios relevantes: los valores de los meses con pocos pedidos (2016-09 a 2016-12 y 2018-09 a 2018-10) se calculan sobre muy pocos pedidos y son poco comparables con el resto.